# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [4]:
with open('aaaa.txt', 'r', encoding='utf-8') as f:
    data = f.read()

print(data)


FileNotFoundError: [Errno 2] No such file or directory: 'aaaa.txt'

In [1]:
!nvidia-smi

Fri Apr  3 00:29:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
os.getcwd()

'/content'

In [3]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip install "peft>=0.14.0" "bitsandbytes>=0.46.1" datasets pillow pandas sentencepiece einops --upgrade
#!pip install flash-attn --no-build-isolation

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-n5qta6ji
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-n5qta6ji
  Resolved https://github.com/huggingface/transformers to commit edaac7db98e34208209fd67d8c66781b8c2e4a53
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
!pip install flash-attn --no-build-isolation

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


In [ ]:
!pip install -v flash-attn --no-build-isolation

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Running command python setup.py egg_info
  /usr/local/lib/python3.12/dist-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  /usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
  !!

          ********************************************************************************
          Requirements should be satisfied by a PEP 517 installer.
          If you are using pip, you can try `pip install --use-pep517`.
          ********************************************************************************

  !!
    dist.fetch_build_eggs(dist.s

In [4]:
import torch
import transformers
import accelerate
import peft
import bitsandbytes

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.10.0+cu128
transformers: 5.5.0.dev0
accelerate: 1.13.0
peft: 0.18.1
bitsandbytes: 0.49.2
cuda available: True


In [ ]:
import flash_attn
print("flash-attn 설치됨")

ModuleNotFoundError: No module named 'flash_attn'

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [ ]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import json
import math
import torch
import pandas as pd
from datetime import datetime

EXP_NAME = "qwen_letter_cv_v1"
RUN_NAME = f"{EXP_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

SAVE_ROOT = f"/content/drive/MyDrive/ai_challenge/{RUN_NAME}"
CV_ROOT = os.path.join(SAVE_ROOT, "cv_runs")

os.makedirs(SAVE_ROOT, exist_ok=True)
os.makedirs(CV_ROOT, exist_ok=True)

print("SAVE_ROOT:", SAVE_ROOT)
print("CV_ROOT  :", CV_ROOT)

SAVE_ROOT: /content/drive/MyDrive/ai_challenge/qwen_letter_cv_v1_20260403_003218
CV_ROOT  : /content/drive/MyDrive/ai_challenge/qwen_letter_cv_v1_20260403_003218/cv_runs


In [3]:
# 압축 해제
!unzip "/content/260401_15_2_ai_데이터배포용.zip" -d "/content/"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정 - 수정 완료

In [6]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from sklearn.model_selection import StratifiedKFold
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from contextlib import nullcontext

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

#from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
#from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 448
MAX_NEW_TOKENS = 4
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# 학습데이터 200개만 추출
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor - 수정 완료

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [7]:
USE_QLORA = True
USE_FLASH_ATTN = False   # 사용 안 함

def get_bnb_config():
    if not USE_QLORA:
        return None

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

def build_processor():
    return AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=IMAGE_SIZE * IMAGE_SIZE,
        max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    )

def build_train_model():
    processor = build_processor()

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=get_bnb_config(),
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",   # eager -> sdpa 변경
    )

    if USE_QLORA:
        base_model = prepare_model_for_kbit_training(base_model)

    base_model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(base_model, lora_config)
    return processor, model

def load_infer_model(adapter_dir):
    processor = build_processor()

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=get_bnb_config(),
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
    )

    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return processor, model

processor, model = build_train_model()
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [8]:
def make_qtype(question: str):
    q = str(question)
    if "몇 개" in q or "개수" in q:
        return "count"
    if "색" in q:
        return "color"
    if "재질" in q:
        return "material"
    if "모양" in q:
        return "shape"
    return "object"


SYSTEM_INSTRUCT = (
    "You are a precise visual multiple-choice question answering assistant. "
    "You must choose the single best option based only on the image and the question. "
    "Return exactly one lowercase letter among a, b, c, or d. "
    "Do not output any other text."
)


def build_mc_prompt(question, a, b, c, d):
    qtype = make_qtype(question)

    guide_map = {
        "count": "이미지에서 실제로 보이는 대상만 세세요. 겹치거나 일부만 보여도 동일한 물체인지 주의해서 판단하세요.",
        "color": "질문에서 묻는 대상의 색만 판단하세요. 배경색이나 다른 물체 색은 무시하세요.",
        "material": "색이나 용도보다 재질을 우선 판단하세요. 플라스틱, 유리, 금속, 종이 등을 구분하세요.",
        "shape": "질문에서 묻는 대상의 전체적인 형태를 보고 모양을 판단하세요.",
        "object": "질문에서 묻는 대상이 무엇인지 이미지에서 가장 잘 맞는 보기를 고르세요.",
    }

    guide = guide_map[qtype]

    return (
        "다음은 이미지 기반 4지선다 문제입니다.\n"
        f"질문: {question}\n\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        f"지침: {guide}\n"
        "반드시 정답 문자 하나만 출력하세요. "
        "출력 형식은 a 또는 b 또는 c 또는 d 중 하나의 소문자 한 글자만 허용됩니다."
    )

In [9]:
LETTERS = ["a", "b", "c", "d"]

def remap_row_options(row, shuffle=False):
    raw_options = [
        ("a", str(row["a"])),
        ("b", str(row["b"])),
        ("c", str(row["c"])),
        ("d", str(row["d"])),
    ]

    if shuffle:
        random.shuffle(raw_options)

    mapped = {}
    gold_old = str(row["answer"]).strip().lower() if "answer" in row.index else None
    gold_new = None

    for new_letter, (old_letter, text) in zip(LETTERS, raw_options):
        mapped[new_letter] = text
        if gold_old is not None and old_letter == gold_old:
            gold_new = new_letter

    return mapped, gold_new

# Custom Dataset, Collator - 수정 완료

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [10]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True, shuffle_options=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.shuffle_options = shuffle_options

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        options, gold_letter = remap_row_options(
            row,
            shuffle=(self.train and self.shuffle_options)
        )

        user_text = build_mc_prompt(
            q,
            options["a"],
            options["b"],
            options["c"],
            options["d"]
        )

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        gold_text = None
        if self.train:
            gold_text = gold_letter   # 정답 text -> 정답 letter
            messages.append({
                "role": "assistant",
                "content": [{"type": "text", "text": gold_text}]
            })

        return {
            "messages": messages,
            "image": img,
            "gold_text": gold_text,
            "gold_letter": gold_letter
        }


@dataclass
class TrainCollator:
    processor: Any

    def __call__(self, batch):
        input_ids_list = []
        labels_list = []
        attention_masks_list = []
        pixel_values_list = []
        image_grid_thw_list = []
        mm_token_type_ids_list = []

        pad_id = self.processor.tokenizer.pad_token_id

        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            full_text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )

            prompt_messages = messages[:-1]
            prompt_text = self.processor.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            full_enc = self.processor(
                text=[full_text],
                images=[img],
                return_tensors="pt"
            )

            prompt_enc = self.processor(
                text=[prompt_text],
                images=[img],
                return_tensors="pt"
            )

            input_ids = full_enc["input_ids"][0]
            attention_mask = full_enc["attention_mask"][0]
            pixel_values = full_enc["pixel_values"][0]
            image_grid_thw = full_enc["image_grid_thw"][0]
            mm_token_type_ids = full_enc["mm_token_type_ids"][0]

            prompt_len = prompt_enc["input_ids"].shape[1]

            labels = input_ids.clone()
            labels[:prompt_len] = -100

            input_ids_list.append(input_ids)
            labels_list.append(labels)
            attention_masks_list.append(attention_mask)
            pixel_values_list.append(pixel_values)
            image_grid_thw_list.append(image_grid_thw)
            mm_token_type_ids_list.append(mm_token_type_ids)

        max_len = max(x.size(0) for x in input_ids_list)

        batch_input_ids = []
        batch_labels = []
        batch_attention_masks = []
        batch_mm_token_type_ids = []

        for input_ids, labels, attention_mask, mm_token_type_ids in zip(
            input_ids_list, labels_list, attention_masks_list, mm_token_type_ids_list
        ):
            pad_len = max_len - input_ids.size(0)

            batch_input_ids.append(
                torch.cat([input_ids, torch.full((pad_len,), pad_id, dtype=input_ids.dtype)])
            )
            batch_labels.append(
                torch.cat([labels, torch.full((pad_len,), -100, dtype=labels.dtype)])
            )
            batch_attention_masks.append(
                torch.cat([attention_mask, torch.zeros(pad_len, dtype=attention_mask.dtype)])
            )
            batch_mm_token_type_ids.append(
                torch.cat([mm_token_type_ids, torch.zeros(pad_len, dtype=mm_token_type_ids.dtype)])
            )

        return {
            "input_ids": torch.stack(batch_input_ids),
            "labels": torch.stack(batch_labels),
            "attention_mask": torch.stack(batch_attention_masks),
            "pixel_values": torch.stack(pixel_values_list),
            "image_grid_thw": torch.stack(image_grid_thw_list),
            "mm_token_type_ids": torch.stack(batch_mm_token_type_ids),
        }

# DataLoader - 수정 완료

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [11]:
def make_qtype(q):
    q = str(q)
    if "몇 개" in q or "개수" in q:
        return "count"
    if "색" in q:
        return "color"
    if "재질" in q:
        return "material"
    if "모양" in q:
        return "shape"
    return "object"

train_df["qtype"] = train_df["question"].apply(make_qtype)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_indices = list(skf.split(train_df, train_df["qtype"]))

def build_fold_dataloaders(fold_id, processor):
    train_idx, valid_idx = fold_indices[fold_id]

    train_subset = train_df.iloc[train_idx].reset_index(drop=True)
    valid_subset = train_df.iloc[valid_idx].reset_index(drop=True)

    train_ds = VQAMCDataset(
        train_subset,
        processor,
        train=True,
        shuffle_options=True     # 학습 때만 셔플
    )

    valid_ds = VQAMCDataset(
        valid_subset,
        processor,
        train=True,
        shuffle_options=False
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=1,
        shuffle=True,
        collate_fn=TrainCollator(processor),
        num_workers=2,
        pin_memory=True,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=TrainCollator(processor),
        num_workers=2,
        pin_memory=True,
    )

    return train_subset, valid_subset, train_loader, valid_loader

for fold_id, (tr_idx, va_idx) in enumerate(fold_indices):
    print(f"fold {fold_id}: train={len(tr_idx)}, valid={len(va_idx)}")

fold 0: train=4058, valid=1015
fold 1: train=4058, valid=1015
fold 2: train=4058, valid=1015
fold 3: train=4059, valid=1014
fold 4: train=4059, valid=1014


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


In [12]:
def save_log(logs, save_path):
    df = pd.DataFrame(logs)
    df.to_csv(save_path, index=False)

def save_training_state(save_dir, epoch, global_step, best_val_acc, logs):
    state = {
        "epoch": epoch,
        "global_step": global_step,
        "best_val_acc": best_val_acc,
        "logs": logs,
    }
    with open(os.path.join(save_dir, "training_state.json"), "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

def save_checkpoint(
    save_dir,
    model,
    processor,
    optimizer,
    scheduler,
    epoch,
    global_step,
    best_val_acc,
    logs
):
    os.makedirs(save_dir, exist_ok=True)

    model.save_pretrained(save_dir)
    processor.save_pretrained(save_dir)

    torch.save({
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "global_step": global_step,
        "best_val_acc": best_val_acc,
        "logs": logs,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }, os.path.join(save_dir, "trainer_state.pt"))

    save_training_state(save_dir, epoch, global_step, best_val_acc, logs)

def load_checkpoint_if_exists(save_dir, model, optimizer=None, scheduler=None):
    trainer_state_path = os.path.join(save_dir, "trainer_state.pt")

    if not os.path.exists(trainer_state_path):
        print("체크포인트 없음. 처음부터 학습 시작.")
        return 0, 0, -1.0, []

    print(f"체크포인트 로드: {save_dir}")
    state = torch.load(trainer_state_path, map_location="cpu")

    if optimizer is not None and state.get("optimizer") is not None:
        optimizer.load_state_dict(state["optimizer"])

    if scheduler is not None and state.get("scheduler") is not None:
        scheduler.load_state_dict(state["scheduler"])

    start_epoch = state.get("epoch", 0)
    global_step = state.get("global_step", 0)
    best_val_acc = state.get("best_val_acc", -1.0)
    logs = state.get("logs", [])

    return start_epoch, global_step, best_val_acc, logs

In [13]:
from tqdm.auto import tqdm
from contextlib import nullcontext
from PIL import Image

def calc_option_scores(model, processor, row, device):
    img = Image.open(row["path"]).convert("RGB")

    q = str(row["question"])
    options = {
        "a": str(row["a"]),
        "b": str(row["b"]),
        "c": str(row["c"]),
        "d": str(row["d"]),
    }

    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(q, options["a"], options["b"], options["c"], options["d"])}
        ]}
    ]

    prompt_text = processor.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_inputs = processor(
        text=[prompt_text],
        images=[img],
        return_tensors="pt"
    ).to(device)

    prompt_len = prompt_inputs["input_ids"].shape[1]
    amp_ctx = torch.amp.autocast("cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

    score_dict = {}

    for letter in LETTERS:
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": letter}]}
        ]

        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        full_inputs = processor(
            text=[full_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        input_ids = full_inputs["input_ids"]
        attention_mask = full_inputs["attention_mask"]
        pixel_values = full_inputs["pixel_values"]
        image_grid_thw = full_inputs["image_grid_thw"]
        mm_token_type_ids = full_inputs["mm_token_type_ids"]

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad(), amp_ctx:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                mm_token_type_ids=mm_token_type_ids,
                labels=labels
            )

        score_dict[letter] = -outputs.loss.item()

    return score_dict


def score_options(model, processor, row, device):
    score_dict = calc_option_scores(model, processor, row, device)
    return max(score_dict, key=score_dict.get)


def evaluate_val_loss(model, valid_loader, device):
    model.eval()
    total_loss = 0.0
    total_count = 0
    amp_ctx = torch.amp.autocast("cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Valid Loss", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}

            with amp_ctx:
                outputs = model(**batch)
                loss = outputs.loss

            bs = batch["input_ids"].size(0)
            total_loss += loss.item() * bs
            total_count += bs

    return total_loss / max(total_count, 1)


def evaluate_val_acc(model, processor, valid_subset, device):
    model.eval()
    correct = 0
    total = 0
    qtype_stats = {}

    for i in tqdm(range(len(valid_subset)), desc="Valid Acc", leave=False):
        row = valid_subset.iloc[i]
        pred = score_options(model, processor, row, device)
        gold = str(row["answer"]).strip().lower()
        qtype = row["qtype"]

        correct += int(pred == gold)
        total += 1

        if qtype not in qtype_stats:
            qtype_stats[qtype] = {"correct": 0, "total": 0}
        qtype_stats[qtype]["correct"] += int(pred == gold)
        qtype_stats[qtype]["total"] += 1

    qtype_acc = {
        k: v["correct"] / max(v["total"], 1)
        for k, v in qtype_stats.items()
    }

    return correct / max(total, 1), qtype_acc

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [14]:
EPOCHS = 2
GRAD_ACCUM = 8
SAVE_EVERY_STEPS = 300
MAX_GRAD_NORM = 1.0

LR = 5e-5
WEIGHT_DECAY = 0.01

In [15]:
import math
import torch
from transformers import get_cosine_schedule_with_warmup

def build_optimizer_scheduler(model, train_loader):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    num_training_steps = EPOCHS * num_update_steps_per_epoch
    num_warmup_steps = int(num_training_steps * 0.05)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    return optimizer, scheduler

# 바로 아래 쉘은 안씀

In [ ]:
import torch
from PIL import Image

def score_options(row):
    img = Image.open(row["path"]).convert("RGB")

    q = str(row["question"])
    options = {
        "a": str(row["a"]),
        "b": str(row["b"]),
        "c": str(row["c"]),
        "d": str(row["d"]),
    }

    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(q, options["a"], options["b"], options["c"], options["d"])}
        ]}
    ]

    best_letter = None
    best_score = -1e18

    for letter, answer_text in options.items():
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": answer_text}]}
        ]

        prompt_text = processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )
        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        full_inputs = processor(
            text=[full_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        prompt_inputs = processor(
            text=[prompt_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        input_ids = full_inputs["input_ids"]
        attention_mask = full_inputs["attention_mask"]
        pixel_values = full_inputs["pixel_values"]
        image_grid_thw = full_inputs["image_grid_thw"]
        mm_token_type_ids = full_inputs["mm_token_type_ids"]

        prompt_len = prompt_inputs["input_ids"].shape[1]

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                mm_token_type_ids=mm_token_type_ids,
                labels=labels
            )

        score = -outputs.loss.item()

        if score > best_score:
            best_score = score
            best_letter = letter

    return best_letter

# 비상임


In [ ]:
val_loss = evaluate_val_loss(model, valid_loader, device)
val_acc, qtype_acc = evaluate_val_acc(model, processor, valid_subset, device)

print(f"val_loss: {val_loss:.4f}")
print(f"val_acc : {val_acc:.4f}")
print("qtype_acc:", qtype_acc)

Valid Loss:   0%|          | 0/508 [00:00<?, ?it/s]

/tmp/ipykernel_42057/1366771142.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  Fi

Valid Acc:   0%|          | 0/508 [00:00<?, ?it/s]

val_loss: 0.1372
val_acc : 0.8839


In [ ]:
row_log = {
    "epoch": 1,
    "global_step": global_step,
    "train_loss": avg_train_loss,
    "val_loss": val_loss,
    "val_acc": val_acc,
    "best_val_acc_before": best_val_acc
}
logs.append(row_log)

if val_acc > best_val_acc:
    best_val_acc = val_acc
    save_checkpoint(
        BEST_DIR,
        model,
        processor,
        optimizer,
        scheduler,
        epoch=1,
        global_step=global_step,
        best_val_acc=best_val_acc,
        logs=logs
    )
    print(f"[BEST 갱신] val_acc={val_acc:.4f}")

save_checkpoint(
    LAST_DIR,
    model,
    processor,
    optimizer,
    scheduler,
    epoch=1,
    global_step=global_step,
    best_val_acc=best_val_acc,
    logs=logs
)
save_log(logs, os.path.join(SAVE_ROOT, "train_log.csv"))

[BEST 갱신] val_acc=0.8839


In [ ]:
import os

print("BEST_DIR exists:", os.path.exists(BEST_DIR))
print("LAST_DIR exists:", os.path.exists(LAST_DIR))
print("train_log exists:", os.path.exists(os.path.join(SAVE_ROOT, "train_log.csv")))

print("\nBEST_DIR files:")
print(os.listdir(BEST_DIR))

print("\nLAST_DIR files:")
print(os.listdir(LAST_DIR))

BEST_DIR exists: True
LAST_DIR exists: True
train_log exists: True

BEST_DIR files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json', 'trainer_state.pt', 'training_state.json']

LAST_DIR files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json', 'trainer_state.pt', 'training_state.json']


# 학습 코드

In [ ]:
from tqdm.auto import tqdm
import torch.nn.utils as nn_utils
import gc

CV_ROOT = os.path.join(SAVE_ROOT, "cv_runs")
os.makedirs(CV_ROOT, exist_ok=True)

RUN_FOLDS = [1,2]   # 테스트 후 [0,1,2,3,4]로 변경
RESUME = False    # 현재 구조에서는 False 권장

for fold_id in RUN_FOLDS:
    print(f"\n{'='*80}")
    print(f"Fold {fold_id+1}/5")
    print(f"{'='*80}")

    fold_root = os.path.join(CV_ROOT, f"fold_{fold_id}")
    BEST_DIR = os.path.join(fold_root, "best_model")
    LAST_DIR = os.path.join(fold_root, "last_checkpoint")
    STEP_DIR = os.path.join(fold_root, "step_checkpoints")

    os.makedirs(fold_root, exist_ok=True)
    os.makedirs(BEST_DIR, exist_ok=True)
    os.makedirs(LAST_DIR, exist_ok=True)
    os.makedirs(STEP_DIR, exist_ok=True)

    processor, model = build_train_model()
    model.print_trainable_parameters()

    train_subset, valid_subset, train_loader, valid_loader = build_fold_dataloaders(fold_id, processor)
    optimizer, scheduler = build_optimizer_scheduler(model, train_loader)

    logs = []

    if RESUME:
        start_epoch, global_step, best_val_acc, logs = load_checkpoint_if_exists(
            LAST_DIR, model, optimizer, scheduler
        )
    else:
        start_epoch, global_step, best_val_acc, logs = 0, 0, -1.0, []

    print(f"start_epoch={start_epoch}, global_step={global_step}, best_val_acc={best_val_acc:.4f}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        train_loss_sum = 0.0
        train_loss_count = 0

        pbar = tqdm(train_loader, desc=f"Fold {fold_id} Epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss
                loss_for_backward = loss / GRAD_ACCUM

            loss_for_backward.backward()

            running_loss += loss.item()
            train_loss_sum += loss.item()
            train_loss_count += 1

            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                nn_utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1
                avg_running = running_loss / GRAD_ACCUM
                pbar.set_postfix({
                    "step_loss": f"{avg_running:.4f}",
                    "gstep": global_step
                })
                running_loss = 0.0

                if global_step % SAVE_EVERY_STEPS == 0:
                    step_save_dir = os.path.join(STEP_DIR, f"step_{global_step}")
                    save_checkpoint(
                        step_save_dir,
                        model,
                        processor,
                        optimizer,
                        scheduler,
                        epoch=epoch,
                        global_step=global_step,
                        best_val_acc=best_val_acc,
                        logs=logs
                    )

                    save_checkpoint(
                        LAST_DIR,
                        model,
                        processor,
                        optimizer,
                        scheduler,
                        epoch=epoch,
                        global_step=global_step,
                        best_val_acc=best_val_acc,
                        logs=logs
                    )

        avg_train_loss = train_loss_sum / max(train_loss_count, 1)

        val_loss = evaluate_val_loss(model, valid_loader, device)
        val_acc, qtype_acc = evaluate_val_acc(model, processor, valid_subset, device)

        row_log = {
            "fold": fold_id,
            "epoch": epoch + 1,
            "global_step": global_step,
            "train_loss": avg_train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "best_val_acc_before": best_val_acc
        }
        logs.append(row_log)

        print(f"\n[Fold {fold_id} Epoch {epoch+1}]")
        print(f"train_loss: {avg_train_loss:.4f}")
        print(f"val_loss  : {val_loss:.4f}")
        print(f"val_acc   : {val_acc:.4f}")
        print(f"qtype_acc : {qtype_acc}")
        print(f"best_acc  : {best_val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(
                BEST_DIR,
                model,
                processor,
                optimizer,
                scheduler,
                epoch=epoch + 1,
                global_step=global_step,
                best_val_acc=best_val_acc,
                logs=logs
            )
            print(f"[BEST 갱신] fold={fold_id}, val_acc={val_acc:.4f}")

        save_checkpoint(
            LAST_DIR,
            model,
            processor,
            optimizer,
            scheduler,
            epoch=epoch + 1,
            global_step=global_step,
            best_val_acc=best_val_acc,
            logs=logs
        )
        save_log(logs, os.path.join(fold_root, "train_log.csv"))

    del model, processor, train_loader, valid_loader, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()


Fold 2/5


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954
start_epoch=0, global_step=0, best_val_acc=-1.0000


Fold 1 Epoch 1/2:   0%|          | 0/4058 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a46d54ba340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

KeyboardInterrupt: 

# 로컬 저장 방식

In [16]:
import os

test_dir = "/content/qwen_letter_cv_local/test_check"
os.makedirs(test_dir, exist_ok=True)

test_path = os.path.join(test_dir, "ok.txt")
with open(test_path, "w", encoding="utf-8") as f:
    f.write("local save works")

print("exists:", os.path.exists(test_path))
print("saved to:", test_path)

exists: True
saved to: /content/qwen_letter_cv_local/test_check/ok.txt


In [18]:
from tqdm.auto import tqdm
import torch.nn.utils as nn_utils
import gc
import os
import shutil
from google.colab import files

# =========================
# 로컬 저장 경로
# =========================
LOCAL_SAVE_ROOT = "/content/qwen_letter_cv_local3"
CV_ROOT = os.path.join(LOCAL_SAVE_ROOT, "cv_runs")
os.makedirs(CV_ROOT, exist_ok=True)

# 지금 돌릴 fold 지정
RUN_FOLDS = [2]
RESUME = False

# 용량 절약용: step checkpoint 저장 끄기
ENABLE_STEP_CHECKPOINTS = False

print("LOCAL_SAVE_ROOT:", LOCAL_SAVE_ROOT)
print("CV_ROOT:", CV_ROOT)

for fold_id in RUN_FOLDS:
    print(f"\n{'='*80}")
    print(f"Fold {fold_id+1}/5")
    print(f"{'='*80}")

    fold_root = os.path.join(CV_ROOT, f"fold_{fold_id}")
    BEST_DIR = os.path.join(fold_root, "best_model")
    LAST_DIR = os.path.join(fold_root, "last_checkpoint")
    STEP_DIR = os.path.join(fold_root, "step_checkpoints")

    os.makedirs(fold_root, exist_ok=True)
    os.makedirs(BEST_DIR, exist_ok=True)
    os.makedirs(LAST_DIR, exist_ok=True)
    if ENABLE_STEP_CHECKPOINTS:
        os.makedirs(STEP_DIR, exist_ok=True)

    processor, model = build_train_model()
    model.print_trainable_parameters()

    train_subset, valid_subset, train_loader, valid_loader = build_fold_dataloaders(fold_id, processor)
    optimizer, scheduler = build_optimizer_scheduler(model, train_loader)

    logs = []

    if RESUME:
        start_epoch, global_step, best_val_acc, logs = load_checkpoint_if_exists(
            LAST_DIR, model, optimizer, scheduler
        )
    else:
        start_epoch, global_step, best_val_acc, logs = 0, 0, -1.0, []

    print(f"start_epoch={start_epoch}, global_step={global_step}, best_val_acc={best_val_acc:.4f}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0
        train_loss_sum = 0.0
        train_loss_count = 0

        pbar = tqdm(train_loader, desc=f"Fold {fold_id} Epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss

            loss_for_backward = loss / GRAD_ACCUM
            loss_for_backward.backward()

            running_loss += loss.item()
            train_loss_sum += loss.item()
            train_loss_count += 1

            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                nn_utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                avg_running = running_loss / GRAD_ACCUM
                pbar.set_postfix({
                    "step_loss": f"{avg_running:.4f}",
                    "gstep": global_step
                })
                running_loss = 0.0

                if ENABLE_STEP_CHECKPOINTS and global_step % SAVE_EVERY_STEPS == 0:
                    step_save_dir = os.path.join(STEP_DIR, f"step_{global_step}")
                    save_checkpoint(
                        step_save_dir,
                        model,
                        processor,
                        optimizer,
                        scheduler,
                        epoch=epoch,
                        global_step=global_step,
                        best_val_acc=best_val_acc,
                        logs=logs
                    )

                save_checkpoint(
                    LAST_DIR,
                    model,
                    processor,
                    optimizer,
                    scheduler,
                    epoch=epoch,
                    global_step=global_step,
                    best_val_acc=best_val_acc,
                    logs=logs
                )

        avg_train_loss = train_loss_sum / max(train_loss_count, 1)
        val_loss = evaluate_val_loss(model, valid_loader, device)
        val_acc, qtype_acc = evaluate_val_acc(model, processor, valid_subset, device)

        row_log = {
            "fold": fold_id,
            "epoch": epoch + 1,
            "global_step": global_step,
            "train_loss": avg_train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "best_val_acc_before": best_val_acc
        }
        logs.append(row_log)

        print(f"\n[Fold {fold_id} Epoch {epoch+1}]")
        print(f"train_loss: {avg_train_loss:.4f}")
        print(f"val_loss : {val_loss:.4f}")
        print(f"val_acc : {val_acc:.4f}")
        print(f"qtype_acc : {qtype_acc}")
        print(f"best_acc : {best_val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(
                BEST_DIR,
                model,
                processor,
                optimizer,
                scheduler,
                epoch=epoch + 1,
                global_step=global_step,
                best_val_acc=best_val_acc,
                logs=logs
            )
            print(f"[BEST 갱신] fold={fold_id}, val_acc={val_acc:.4f}")

        save_checkpoint(
            LAST_DIR,
            model,
            processor,
            optimizer,
            scheduler,
            epoch=epoch + 1,
            global_step=global_step,
            best_val_acc=best_val_acc,
            logs=logs
        )

        save_log(logs, os.path.join(fold_root, "train_log.csv"))

    del model, processor, train_loader, valid_loader, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# =========================
# 학습 종료 후 zip 생성 + 다운로드
# =========================
archive_base = f"/content/qwen_letter_cv_local_folds_{'_'.join(map(str, RUN_FOLDS))}"
archive_path = shutil.make_archive(archive_base, "zip", LOCAL_SAVE_ROOT)

print("ZIP saved:", archive_path)
files.download(archive_path)

LOCAL_SAVE_ROOT: /content/qwen_letter_cv_local3
CV_ROOT: /content/qwen_letter_cv_local3/cv_runs

Fold 3/5


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954
start_epoch=0, global_step=0, best_val_acc=-1.0000


Fold 2 Epoch 1/2:   0%|          | 0/4058 [00:00<?, ?it/s]

...

Valid Loss:   0%|          | 0/1015 [00:00<?, ?it/s]

Valid Acc:   0%|          | 0/1015 [00:00<?, ?it/s]


[Fold 2 Epoch 2]
train_loss: 0.1903
val_loss : 0.2213
val_acc : 0.8463
qtype_acc : {'material': 0.9255663430420712, 'color': 0.8837209302325582, 'object': 0.9481481481481482, 'count': 0.6876790830945558, 'shape': 1.0}
best_acc : 0.8335
[BEST 갱신] fold=2, val_acc=0.8463
ZIP saved: /content/qwen_letter_cv_local_folds_2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# count sp

In [19]:
from contextlib import nullcontext
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import torch
import os
import math

COUNT_SYSTEM_INSTRUCT = (
    "You are a precise visual multiple-choice counting assistant. "
    "Count only the real visible recyclable objects asked in the question. "
    "Ignore printed pictures, reflections, background patterns, and unrelated objects. "
    "Return exactly one lowercase letter among a, b, c, or d."
)

def build_count_mc_prompt(question, a, b, c, d):
    return (
        "다음은 이미지 기반 개수 세기 4지선다 문제입니다.\n"
        f"질문: {question}\n\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        "지침:\n"
        "- 실제로 보이는 재활용 가능한 대상만 세세요.\n"
        "- 배경 그림, 라벨 그림, 반사된 물체는 세지 마세요.\n"
        "- 일부만 보여도 서로 다른 실제 물체면 각각 세세요.\n"
        "- 반드시 정답 문자 하나만 출력하세요.\n"
        "출력 형식은 a 또는 b 또는 c 또는 d 중 하나의 소문자 한 글자만 허용됩니다."
    )

class CountSpecialistDataset(Dataset):
    def __init__(self, df, processor, train=True, shuffle_options=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        self.shuffle_options = shuffle_options

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        options, gold_letter = remap_row_options(
            row,
            shuffle=(self.train and self.shuffle_options)
        )

        user_text = build_count_mc_prompt(
            q,
            options["a"],
            options["b"],
            options["c"],
            options["d"]
        )

        messages = [
            {"role": "system", "content": [{"type": "text", "text": COUNT_SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        gold_text = None
        if self.train:
            gold_text = gold_letter
            messages.append({
                "role": "assistant",
                "content": [{"type": "text", "text": gold_text}]
            })

        return {
            "messages": messages,
            "image": img,
            "gold_text": gold_text,
            "gold_letter": gold_letter
        }

def build_count_specialist_dataloaders(fold_id, processor):
    train_idx, valid_idx = fold_indices[fold_id]

    train_subset = train_df.iloc[train_idx].copy()
    valid_subset = train_df.iloc[valid_idx].copy()

    train_subset = train_subset[train_subset["qtype"] == "count"].reset_index(drop=True)
    valid_subset = valid_subset[valid_subset["qtype"] == "count"].reset_index(drop=True)

    print(f"[count specialist fold {fold_id}] train={len(train_subset)}, valid={len(valid_subset)}")

    train_ds = CountSpecialistDataset(
        train_subset,
        processor,
        train=True,
        shuffle_options=True
    )

    valid_ds = CountSpecialistDataset(
        valid_subset,
        processor,
        train=True,
        shuffle_options=False
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=1,
        shuffle=True,
        collate_fn=TrainCollator(processor),
        num_workers=0,
        pin_memory=True,
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=TrainCollator(processor),
        num_workers=0,
        pin_memory=True,
    )

    return train_subset, valid_subset, train_loader, valid_loader

def build_count_optimizer_scheduler(model, train_loader, count_epochs, count_lr, weight_decay=0.01):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=count_lr,
        weight_decay=weight_decay
    )

    num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    num_training_steps = count_epochs * num_update_steps_per_epoch
    num_warmup_steps = max(1, int(num_training_steps * 0.05))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    return optimizer, scheduler

def calc_count_option_scores(model, processor, row, device):
    img = Image.open(row["path"]).convert("RGB")

    q = str(row["question"])
    options = {
        "a": str(row["a"]),
        "b": str(row["b"]),
        "c": str(row["c"]),
        "d": str(row["d"]),
    }

    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": COUNT_SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_count_mc_prompt(q, options["a"], options["b"], options["c"], options["d"])}
        ]}
    ]

    prompt_text = processor.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    prompt_inputs = processor(
        text=[prompt_text],
        images=[img],
        return_tensors="pt"
    ).to(device)

    prompt_len = prompt_inputs["input_ids"].shape[1]
    amp_ctx = torch.amp.autocast("cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

    score_dict = {}

    for letter in LETTERS:
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": letter}]}
        ]

        full_text = processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        full_inputs = processor(
            text=[full_text],
            images=[img],
            return_tensors="pt"
        ).to(device)

        input_ids = full_inputs["input_ids"]
        attention_mask = full_inputs["attention_mask"]
        pixel_values = full_inputs["pixel_values"]
        image_grid_thw = full_inputs["image_grid_thw"]
        mm_token_type_ids = full_inputs["mm_token_type_ids"]

        labels = input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad(), amp_ctx:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                mm_token_type_ids=mm_token_type_ids,
                labels=labels
            )

        score_dict[letter] = -outputs.loss.item()

    return score_dict

def evaluate_count_val_acc(model, processor, valid_subset, device):
    model.eval()
    correct = 0
    total = 0

    for i in tqdm(range(len(valid_subset)), desc="Count Valid Acc", leave=False):
        row = valid_subset.iloc[i]
        score_dict = calc_count_option_scores(model, processor, row, device)
        pred = max(score_dict, key=score_dict.get)
        gold = str(row["answer"]).strip().lower()

        correct += int(pred == gold)
        total += 1

    return correct / max(total, 1)

In [20]:
from peft import PeftModel

def load_fold_adapter_for_count_specialist(fold_best_dir):
    processor = build_processor()

    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=get_bnb_config(),
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
    )

    if USE_QLORA:
        base_model = prepare_model_for_kbit_training(base_model)

    base_model.gradient_checkpointing_enable()

    model = PeftModel.from_pretrained(base_model, fold_best_dir, is_trainable=True)
    model.train()

    return processor, model

In [23]:
from tqdm.auto import tqdm
import torch.nn.utils as nn_utils
import gc
import os

# ===== 시작점: 일반 fold0 모델 =====
FOLD0_BEST_DIR = "/content/qwen_letter_cv_local3/cv_runs/fold_0/best_model"

# ===== count specialist 데이터는 fold3의 count만 사용 =====
COUNT_SPECIALIST_FOLD = 3
COUNT_EPOCHS = 1
COUNT_LR = 3e-5
COUNT_WEIGHT_DECAY = 0.01

# ===== 저장은 완전히 별도 폴더 =====
COUNT_SAVE_ROOT = "/content/qwen_count_specialist_from_fold0"
COUNT_CV_ROOT = os.path.join(COUNT_SAVE_ROOT, "cv_runs")
os.makedirs(COUNT_CV_ROOT, exist_ok=True)

fold_id = COUNT_SPECIALIST_FOLD
fold_root = os.path.join(COUNT_CV_ROOT, f"fold_{fold_id}_count_from_fold0")
BEST_DIR = os.path.join(fold_root, "best_model")
LAST_DIR = os.path.join(fold_root, "last_checkpoint")

os.makedirs(fold_root, exist_ok=True)
os.makedirs(BEST_DIR, exist_ok=True)
os.makedirs(LAST_DIR, exist_ok=True)

print("Load source fold0 model from:", FOLD0_BEST_DIR)
print("Save count specialist to   :", fold_root)

processor, model = load_fold_adapter_for_count_specialist(FOLD0_BEST_DIR)
model.print_trainable_parameters()

train_subset, valid_subset, train_loader, valid_loader = build_count_specialist_dataloaders(fold_id, processor)
optimizer, scheduler = build_count_optimizer_scheduler(
    model,
    train_loader,
    count_epochs=COUNT_EPOCHS,
    count_lr=COUNT_LR,
    weight_decay=COUNT_WEIGHT_DECAY
)

logs = []
global_step = 0
best_val_acc = -1.0

print(f"count specialist fold={fold_id}, train={len(train_subset)}, valid={len(valid_subset)}")

for epoch in range(COUNT_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    train_loss_sum = 0.0
    train_loss_count = 0

    pbar = tqdm(train_loader, desc=f"Count Specialist From Fold0 Epoch {epoch+1}/{COUNT_EPOCHS}")

    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss
            loss_for_backward = loss / GRAD_ACCUM

        loss_for_backward.backward()

        running_loss += loss.item()
        train_loss_sum += loss.item()
        train_loss_count += 1

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            nn_utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            avg_running = running_loss / GRAD_ACCUM
            pbar.set_postfix({
                "step_loss": f"{avg_running:.4f}",
                "gstep": global_step
            })
            running_loss = 0.0

    avg_train_loss = train_loss_sum / max(train_loss_count, 1)
    val_loss = evaluate_val_loss(model, valid_loader, device)
    val_acc = evaluate_count_val_acc(model, processor, valid_subset, device)

    row_log = {
        "fold": fold_id,
        "epoch": epoch + 1,
        "global_step": global_step,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "best_val_acc_before": best_val_acc
    }
    logs.append(row_log)

    print(f"\n[Count Specialist From Fold0 Epoch {epoch+1}]")
    print(f"train_loss: {avg_train_loss:.4f}")
    print(f"val_loss  : {val_loss:.4f}")
    print(f"val_acc   : {val_acc:.4f}")
    print(f"best_acc  : {best_val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(
            BEST_DIR,
            model,
            processor,
            optimizer,
            scheduler,
            epoch=epoch + 1,
            global_step=global_step,
            best_val_acc=best_val_acc,
            logs=logs
        )
        print(f"[COUNT BEST 갱신] val_acc={val_acc:.4f}")

    save_checkpoint(
        LAST_DIR,
        model,
        processor,
        optimizer,
        scheduler,
        epoch=epoch + 1,
        global_step=global_step,
        best_val_acc=best_val_acc,
        logs=logs
    )
    save_log(logs, os.path.join(fold_root, "train_log.csv"))

del model, processor, train_loader, valid_loader, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()

Load source fold0 model from: /content/qwen_letter_cv_local3/cv_runs/fold_0/best_model
Save count specialist to   : /content/qwen_count_specialist_from_fold0/cv_runs/fold_3_count_from_fold0


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954
[count specialist fold 3] train=1396, valid=349
count specialist fold=3, train=1396, valid=349


Count Specialist From Fold0 Epoch 1/1:   0%|          | 0/1396 [00:00<?, ?it/s]

Valid Loss:   0%|          | 0/349 [00:00<?, ?it/s]

Count Valid Acc:   0%|          | 0/349 [00:00<?, ?it/s]


[Count Specialist From Fold0 Epoch 1]
train_loss: 0.3073
val_loss  : 0.2894
val_acc   : 0.7479
best_acc  : -1.0000
[COUNT BEST 갱신] val_acc=0.7479


# 추론
1. 가중치별 결과값 (x, 0.2, 0.3, 0.5, 0.7)

2. 하나만 하는 버전

3. 시간상 fold0 + count


In [25]:
import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm

GENERAL_CV_ROOT = "/content/qwen_letter_cv_local3/cv_runs"
GENERAL_FOLDS = [0, 1, 2]

COUNT_SPECIALIST_DIR = "/content/qwen_count_specialist_from_fold0/cv_runs/fold_3_count_from_fold0/best_model"

# 시험할 weight들
COUNT_SPECIALIST_WEIGHTS = [0.2, 0.3, 0.5, 0.7]

def has_adapter(path):
    return os.path.exists(os.path.join(path, "adapter_config.json"))

available_general_folds = []
for fold_id in GENERAL_FOLDS:
    fold_best_dir = os.path.join(GENERAL_CV_ROOT, f"fold_{fold_id}", "best_model")
    if has_adapter(fold_best_dir):
        available_general_folds.append(fold_id)
    else:
        print(f"skip general fold {fold_id}: {fold_best_dir} not found")

if len(available_general_folds) == 0:
    raise ValueError("일반 앙상블용 fold 모델이 없습니다.")

use_count_specialist = has_adapter(COUNT_SPECIALIST_DIR)
print("general folds:", available_general_folds)
print("use_count_specialist:", use_count_specialist)

# 1) general ensemble score 계산
general_score_sums = [
    {letter: 0.0 for letter in LETTERS}
    for _ in range(len(test_df))
]

for fold_id in available_general_folds:
    fold_best_dir = os.path.join(GENERAL_CV_ROOT, f"fold_{fold_id}", "best_model")
    print(f"\nLoad general fold {fold_id} from: {fold_best_dir}")

    processor, model = load_infer_model(fold_best_dir)

    for i in tqdm(range(len(test_df)), desc=f"General Fold {fold_id} Inference", unit="sample"):
        row = test_df.iloc[i]
        score_dict = calc_option_scores(model, processor, row, device)

        for letter in LETTERS:
            general_score_sums[i][letter] += score_dict[letter]

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

general_score_avgs = []
num_general = len(available_general_folds)
for i in range(len(test_df)):
    general_score_avgs.append({
        letter: general_score_sums[i][letter] / num_general
        for letter in LETTERS
    })

# 2) count specialist score 계산
count_specialist_scores = None

if use_count_specialist:
    count_specialist_scores = [
        {letter: 0.0 for letter in LETTERS}
        for _ in range(len(test_df))
    ]

    print(f"\nLoad count specialist from: {COUNT_SPECIALIST_DIR}")
    count_processor, count_model = load_infer_model(COUNT_SPECIALIST_DIR)

    for i in tqdm(range(len(test_df)), desc="Count Specialist Inference", unit="sample"):
        row = test_df.iloc[i]
        qtype = make_qtype(row["question"])

        if qtype == "count":
            score_dict = calc_count_option_scores(count_model, count_processor, row, device)
            count_specialist_scores[i] = score_dict

    del count_model, count_processor
    gc.collect()
    torch.cuda.empty_cache()

# 3) general only 제출 파일
preds_general_only = []

for i in range(len(test_df)):
    final_scores = general_score_avgs[i]
    pred = max(final_scores, key=final_scores.get)
    preds_general_only.append(pred)

submission_general_only = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds_general_only
})

general_only_path = "/content/submission_general012_only.csv"
submission_general_only.to_csv(general_only_path, index=False)
print(f"Saved {general_only_path}")
print(submission_general_only.head())

# 4) weight별 제출 파일 생성
if use_count_specialist:
    for COUNT_SPECIALIST_WEIGHT in COUNT_SPECIALIST_WEIGHTS:
        preds = []

        for i in range(len(test_df)):
            row = test_df.iloc[i]
            qtype = make_qtype(row["question"])

            if qtype == "count":
                final_scores = {}
                for letter in LETTERS:
                    general_score = general_score_avgs[i][letter]
                    specialist_score = count_specialist_scores[i][letter]
                    final_scores[letter] = (
                        (1.0 - COUNT_SPECIALIST_WEIGHT) * general_score
                        + COUNT_SPECIALIST_WEIGHT * specialist_score
                    )
            else:
                final_scores = general_score_avgs[i]

            pred = max(final_scores, key=final_scores.get)
            preds.append(pred)

        submission = pd.DataFrame({
            "id": test_df["id"],
            "answer": preds
        })

        weight_tag = str(COUNT_SPECIALIST_WEIGHT).replace(".", "")
        save_path = f"/content/submission_general012_plus_count_from_fold0_w{weight_tag}.csv"
        submission.to_csv(save_path, index=False)

        print(f"Saved {save_path}")
        print(submission.head())
else:
    print("count specialist를 찾지 못해서 general only만 저장했습니다.")

general folds: [0, 1, 2]
use_count_specialist: True

Load general fold 0 from: /content/qwen_letter_cv_local3/cv_runs/fold_0/best_model


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

General Fold 0 Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

KeyboardInterrupt: 

In [ ]:
import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm

GENERAL_CV_ROOT = "/content/qwen_letter_cv_local3/cv_runs"
GENERAL_FOLDS = [0, 1, 2]

COUNT_SPECIALIST_DIR = "/content/qwen_count_specialist_from_fold0/cv_runs/fold_3_count_from_fold0/best_model"

COUNT_SPECIALIST_WEIGHT = 0.5

def has_adapter(path):
    return os.path.exists(os.path.join(path, "adapter_config.json"))

available_general_folds = []
for fold_id in GENERAL_FOLDS:
    fold_best_dir = os.path.join(GENERAL_CV_ROOT, f"fold_{fold_id}", "best_model")
    if has_adapter(fold_best_dir):
        available_general_folds.append(fold_id)
    else:
        print(f"skip general fold {fold_id}: {fold_best_dir} not found")

if len(available_general_folds) == 0:
    raise ValueError("일반 앙상블용 fold 모델이 없습니다.")

use_count_specialist = has_adapter(COUNT_SPECIALIST_DIR)
print("general folds:", available_general_folds)
print("use_count_specialist:", use_count_specialist)

general_score_sums = [
    {letter: 0.0 for letter in LETTERS}
    for _ in range(len(test_df))
]

for fold_id in available_general_folds:
    fold_best_dir = os.path.join(GENERAL_CV_ROOT, f"fold_{fold_id}", "best_model")
    print(f"\nLoad general fold {fold_id} from: {fold_best_dir}")

    processor, model = load_infer_model(fold_best_dir)

    for i in tqdm(range(len(test_df)), desc=f"General Fold {fold_id} Inference", unit="sample"):
        row = test_df.iloc[i]
        score_dict = calc_option_scores(model, processor, row, device)

        for letter in LETTERS:
            general_score_sums[i][letter] += score_dict[letter]

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

general_score_avgs = []
num_general = len(available_general_folds)
for i in range(len(test_df)):
    general_score_avgs.append({
        letter: general_score_sums[i][letter] / num_general
        for letter in LETTERS
    })

count_specialist_scores = None

if use_count_specialist:
    count_specialist_scores = [
        {letter: 0.0 for letter in LETTERS}
        for _ in range(len(test_df))
    ]

    print(f"\nLoad count specialist from: {COUNT_SPECIALIST_DIR}")
    count_processor, count_model = load_infer_model(COUNT_SPECIALIST_DIR)

    for i in tqdm(range(len(test_df)), desc="Count Specialist Inference", unit="sample"):
        row = test_df.iloc[i]
        qtype = make_qtype(row["question"])

        if qtype == "count":
            score_dict = calc_count_option_scores(count_model, count_processor, row, device)
            count_specialist_scores[i] = score_dict

    del count_model, count_processor
    gc.collect()
    torch.cuda.empty_cache()

preds = []

for i in range(len(test_df)):
    row = test_df.iloc[i]
    qtype = make_qtype(row["question"])

    if qtype == "count" and use_count_specialist:
        final_scores = {}
        for letter in LETTERS:
            general_score = general_score_avgs[i][letter]
            specialist_score = count_specialist_scores[i][letter]
            final_scores[letter] = (
                (1.0 - COUNT_SPECIALIST_WEIGHT) * general_score
                + COUNT_SPECIALIST_WEIGHT * specialist_score
            )
    else:
        final_scores = general_score_avgs[i]

    pred = max(final_scores, key=final_scores.get)
    preds.append(pred)

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds
})

submission.to_csv("/content/submission_general012_plus_count_from_fold0.csv", index=False)

print("Saved /content/submission_general012_plus_count_from_fold0.csv")
print(submission.head())

In [26]:
import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm

GENERAL_FOLD0_DIR = "/content/qwen_letter_cv_local3/cv_runs/fold_0/best_model"
COUNT_SPECIALIST_DIR = "/content/qwen_count_specialist_from_fold0/cv_runs/fold_3_count_from_fold0/best_model"

COUNT_SPECIALIST_WEIGHTS = [0.2, 0.3, 0.5, 0.7]

def has_adapter(path):
    return os.path.exists(os.path.join(path, "adapter_config.json"))

if not has_adapter(GENERAL_FOLD0_DIR):
    raise ValueError(f"fold0 모델을 찾을 수 없습니다: {GENERAL_FOLD0_DIR}")

use_count_specialist = has_adapter(COUNT_SPECIALIST_DIR)

print("general fold0:", GENERAL_FOLD0_DIR)
print("use_count_specialist:", use_count_specialist)

# 1) fold0 일반 점수 계산
general_scores = [
    {letter: 0.0 for letter in LETTERS}
    for _ in range(len(test_df))
]

print(f"\nLoad general fold0 from: {GENERAL_FOLD0_DIR}")
general_processor, general_model = load_infer_model(GENERAL_FOLD0_DIR)

for i in tqdm(range(len(test_df)), desc="General Fold0 Inference", unit="sample"):
    row = test_df.iloc[i]
    score_dict = calc_option_scores(general_model, general_processor, row, device)
    general_scores[i] = score_dict

del general_model, general_processor
gc.collect()
torch.cuda.empty_cache()

# 2) count specialist 점수 계산
count_specialist_scores = None

if use_count_specialist:
    count_specialist_scores = [
        {letter: 0.0 for letter in LETTERS}
        for _ in range(len(test_df))
    ]

    print(f"\nLoad count specialist from: {COUNT_SPECIALIST_DIR}")
    count_processor, count_model = load_infer_model(COUNT_SPECIALIST_DIR)

    for i in tqdm(range(len(test_df)), desc="Count Specialist Inference", unit="sample"):
        row = test_df.iloc[i]
        qtype = make_qtype(row["question"])

        if qtype == "count":
            score_dict = calc_count_option_scores(count_model, count_processor, row, device)
            count_specialist_scores[i] = score_dict

    del count_model, count_processor
    gc.collect()
    torch.cuda.empty_cache()

# 3) fold0 only 제출 파일
preds_general_only = []

for i in range(len(test_df)):
    final_scores = general_scores[i]
    pred = max(final_scores, key=final_scores.get)
    preds_general_only.append(pred)

submission_general_only = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds_general_only
})

general_only_path = "/content/submission_fold0_only.csv"
submission_general_only.to_csv(general_only_path, index=False)
print(f"Saved {general_only_path}")
print(submission_general_only.head())

# 4) weight별 제출 파일 생성
if use_count_specialist:
    for COUNT_SPECIALIST_WEIGHT in COUNT_SPECIALIST_WEIGHTS:
        preds = []

        for i in range(len(test_df)):
            row = test_df.iloc[i]
            qtype = make_qtype(row["question"])

            if qtype == "count":
                final_scores = {}
                for letter in LETTERS:
                    general_score = general_scores[i][letter]
                    specialist_score = count_specialist_scores[i][letter]
                    final_scores[letter] = (
                        (1.0 - COUNT_SPECIALIST_WEIGHT) * general_score
                        + COUNT_SPECIALIST_WEIGHT * specialist_score
                    )
            else:
                final_scores = general_scores[i]

            pred = max(final_scores, key=final_scores.get)
            preds.append(pred)

        submission = pd.DataFrame({
            "id": test_df["id"],
            "answer": preds
        })

        weight_tag = str(COUNT_SPECIALIST_WEIGHT).replace(".", "")
        save_path = f"/content/submission_fold0_plus_count_w{weight_tag}.csv"
        submission.to_csv(save_path, index=False)

        print(f"Saved {save_path}")
        print(submission.head())
else:
    print("count specialist를 찾지 못해서 fold0 only만 저장했습니다.")

general fold0: /content/qwen_letter_cv_local3/cv_runs/fold_0/best_model
use_count_specialist: True

Load general fold0 from: /content/qwen_letter_cv_local3/cv_runs/fold_0/best_model


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

General Fold0 Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]


Load count specialist from: /content/qwen_count_specialist_from_fold0/cv_runs/fold_3_count_from_fold0/best_model


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Count Specialist Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

Saved /content/submission_fold0_only.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b
Saved /content/submission_fold0_plus_count_w02.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b
Saved /content/submission_fold0_plus_count_w03.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b
Saved /content/submission_fold0_plus_count_w05.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b
Saved /content/submission_fold0_plus_count_w07.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
import os
import gc
import torch
from tqdm.auto import tqdm
import pandas as pd

CV_ROOT = os.path.join(SAVE_ROOT, "cv_runs")
REQUESTED_FOLDS = [0, 1, 2, 3, 4]   # 다 학습했으면 그대로, 아니면 후보만 적어둬도 됨

available_folds = []
for fold_id in REQUESTED_FOLDS:
    fold_best_dir = os.path.join(CV_ROOT, f"fold_{fold_id}", "best_model")
    if os.path.exists(fold_best_dir):
        available_folds.append(fold_id)
    else:
        print(f"skip fold {fold_id}: {fold_best_dir} not found")

if len(available_folds) == 0:
    raise ValueError("사용 가능한 fold 모델이 없습니다. 먼저 학습을 완료했는지 확인하세요.")

print("using folds:", available_folds)

all_score_sums = [
    {letter: 0.0 for letter in LETTERS}
    for _ in range(len(test_df))
]

for fold_id in available_folds:
    fold_best_dir = os.path.join(CV_ROOT, f"fold_{fold_id}", "best_model")
    print(f"\nLoad fold {fold_id} from: {fold_best_dir}")

    processor, model = load_infer_model(fold_best_dir)

    for i in tqdm(range(len(test_df)), desc=f"Fold {fold_id} Inference", unit="sample"):
        row = test_df.iloc[i]
        score_dict = calc_option_scores(model, processor, row, device)

        for letter in LETTERS:
            all_score_sums[i][letter] += score_dict[letter]

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

preds = [max(score_dict, key=score_dict.get) for score_dict in all_score_sums]

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds
})
submission.to_csv("/content/submission.csv", index=False)

print("Saved /content/submission.csv")
print(submission.head())

Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

Saved /content/submission.csv
              id answer
0  test_0001.jpg      d
1  test_0002.jpg      a
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b


In [ ]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
사진에 보이는 재활용 가능한 아이템은 무엇인가요?
(a) 플라스틱 숟가락
(b) 유리병
(c) 종이 포장지
(d) 금속 캔

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
b


# 드라이브 저장 실패 대비 비상 쉘
- 근데 성능은 낮게 나올듯 이걸로 하면

In [24]:
# 압축 해제
!unzip "/content/qwen_letter_cv_local_folds_1.zip" -d "/content/"

Archive:  /content/qwen_letter_cv_local_folds_1.zip
   creating: /content/cv_runs/
   creating: /content/cv_runs/fold_1/
   creating: /content/cv_runs/fold_1/best_model/
   creating: /content/cv_runs/fold_1/last_checkpoint/
  inflating: /content/cv_runs/fold_1/train_log.csv  
  inflating: /content/cv_runs/fold_1/last_checkpoint/tokenizer_config.json  
  inflating: /content/cv_runs/fold_1/last_checkpoint/processor_config.json  
  inflating: /content/cv_runs/fold_1/last_checkpoint/chat_template.jinja  
  inflating: /content/cv_runs/fold_1/last_checkpoint/tokenizer.json  
  inflating: /content/cv_runs/fold_1/last_checkpoint/README.md  
  inflating: /content/cv_runs/fold_1/last_checkpoint/training_state.json  
  inflating: /content/cv_runs/fold_1/last_checkpoint/adapter_config.json  
  inflating: /content/cv_runs/fold_1/last_checkpoint/trainer_state.pt  
  inflating: /content/cv_runs/fold_1/last_checkpoint/adapter_model.safetensors  
  inflating: /content/cv_runs/fold_1/best_model/tokenize